In [1]:
import boto3
session = boto3.Session()

bedrock = session.client("bedrock", region_name="us-east-1")
br = session.client("bedrock-runtime", region_name="us-east-1")

In [5]:
import dspy

region_name = "us-east-1"

lm = dspy.LM(
    model='bedrock/us.anthropic.claude-3-5-sonnet-20241022-v2:0',
    max_tokens=4096
)

In [6]:
dspy.settings.configure(lm=lm)

In [8]:
math = dspy.ChainOfThought("question -> answer: float")
answer = math(question="Two dice are tossed. What is the probability that the sum equals two?")

In [11]:
print(answer.reasoning)

Let's solve this step by step:
1. For the sum to equal 2, we need both dice to show 1
2. Each die has 6 possible outcomes
3. Total possible outcomes when rolling 2 dice = 6 × 6 = 36
4. Favorable outcome is only one combination: (1,1)
5. Therefore probability = 1/36


In [16]:
for msg in math.history[0]['messages']:
    print(msg['content'])

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (float):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must be a single float value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.
[[ ## question ## ]]
Two dice are tossed. What is the probability that the sum equals two?

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## answer ## ]]` (must be formatted as a valid Python float), and then ending with the marker for `[[ ## completed ## ]]`.


In [ ]:

# Define a module (ChainOfThought) and assign it a signature (return an answer, given a question).
qa = dspy.ChainOfThought('question -> answer')

# # Run with the default LM configured with `dspy.configure` above.
# response = qa(question="How many floors are in the castle David Gregory inherited?")
# print(response.answer)

# Setting the default LM
dspy.configure(lm=dspy.LM('bedrock/us.anthropic.claude-3-5-sonnet-20241022-v2:0'))
response = qa(question="How many floors are in the castle David Gregory inherited?")
print('Sonnet V2', response.answer)

# Change the default LM for a single module.
with dspy.context(lm=dspy.LM('bedrock/us.anthropic.claude-3-haiku-20240307-v1:0')):
    response = qa(question="How many floors are in the castle David Gregory inherited?")
    print('Haiku V1:', response.answer)

Unknown - there is not enough information provided to determine the number of floors in David Gregory's inherited castle.
Sonnet V2 Unknown - there is not enough information provided to determine the number of floors in David Gregory's inherited castle.
Haiku V1: I'm sorry, but I don't have enough information to answer how many floors are in the castle that David Gregory inherited. The question does not provide any details about this castle.


In [19]:
class CheckCitationFaithfulness(dspy.Signature):
    """Verify that the text is based on the provided context."""

    context: str = dspy.InputField(desc="facts here are assumed to be true")
    text: str = dspy.InputField()
    faithfulness: bool = dspy.OutputField()
    evidence: dict[str, list[str]] = dspy.OutputField(desc="Supporting evidence for claims")


task = "context:str, text: str -> faithfulness: bool, evidence: dict[str, list[str]]"

solver = dspy.ChainOfThought(task)

In [ ]:
from pydantic import BaseModel

class ContextQuery(BaseModel):
    context: str
    query: str


class QueryResponse(BaseModel):
    response: str
    evidence: str

solver = dspy.Predict('query: ContextQuery -> response: QueryResponse')

context = "The 21-year-old made seven appearances for the Hammers and netted his only goal for them in a Europa League qualification round match against Andorran side FC Lustrains last season. Lee had two loan spells in League One last term, with Blackpool and then Colchester United. He scored twice for the U's but was unable to save them from relegation. The length of Lee's contract with the promoted Tykes has not been revealed. Find all the latest football transfers on our dedicated page."

text = "Lee scored 3 goals for Colchester United."

query_1 = ContextQuery(
    context=context,
    query=text
)

response = solver(query = query_1)



In [ ]:
class TestObj:
    def __init__(self) -> None:
        pass

    def forward(self, x: int) -> int:
        return x + 1

    def __call__(self, x: int) -> int:
        return self.forward(x)

test_obj = TestObj()

test_obj(1)

2

In [43]:
print(response.response)

response='False' evidence="He scored twice for the U's but was unable to save them from relegation."


In [44]:
for msg in solver.history[0]['messages']:
    print(msg['content'])

Your input fields are:
1. `query` (ContextQuery):
Your output fields are:
1. `response` (QueryResponse):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## query ## ]]
{query}

[[ ## response ## ]]
{response}        # note: the value you produce must adhere to the JSON schema: {"type": "object", "properties": {"evidence": {"type": "string", "title": "Evidence"}, "response": {"type": "string", "title": "Response"}}, "required": ["response", "evidence"], "title": "QueryResponse"}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `query`, produce the fields `response`.
[[ ## query ## ]]
{"context": "The 21-year-old made seven appearances for the Hammers and netted his only goal for them in a Europa League qualification round match against Andorran side FC Lustrains last season. Lee had two loan spells in League One last term, with Blackpool and then Colchester United. He scored twice for 